<a href="https://colab.research.google.com/github/edwardoughton/satellite-image-analysis/blob/main/07_02_ggs416_26_03_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/edwardoughton/satellite-image-analysis/blob/main/07_01_ggs416_26_03_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GGS416 Week 7: YOLO OBB on non-airport overhead objects

This notebook is a variant of the Week 7 lesson that utilizes a YOLO model trained on aerial-imagery classes (`yolo26n-obb.pt`).

The main targets in this version are:
- ship
- harbor
- storage tank
- small vehicle
- large vehicle
- bridge

We keep the same workflow:
- get a recent NAIP image from the Planetary Computer
- clip it to a study area
- tile the image
- run an oriented object detector on each tile
- plot the detections back on the full scene


## Install packages

Run this only if the packages are not already installed in your environment.

In [ ]:
%pip -q install pystac-client planetary-computer requests rasterio matplotlib pandas pillow ultralytics

## What is different in this notebook?

The `yolo26n-obb.pt` model we will utilize is trained on the DOTA dataset, which contains labeled information for aerial classes.

This means we have a better oppotunity for identifying overhead targets in imagery.

We focus on classes such as:
- ship
- harbor
- storage tank
- bridge
- large vehicle
- small vehicle

Important limitation: this is still not the right model for road extraction or building footprint mapping (we will cover that later).


## Import the packages for basic Python work

We start with very common Python packages. These are not specific to geospatial data or machine learning. You should already be familiar with the following:

- `os` helps us work with files and folders.
- `numpy` helps us work with image arrays.
- `pandas` helps us organize detections in tables.


In [ ]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)

## Import the packages for display and plotting

Next we import packages used to show imagery and draw bounding boxes.

- `matplotlib.pyplot` displays images and charts.
- `matplotlib.patches` lets us draw rectangles around detected objects.
- `display` helps us show tables nicely in a notebook.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display

## Import the packages for satellite imagery

These packages are the geospatial part of the workflow.

- `pystac_client` searches the Planetary Computer catalog.
- `planetary_computer` signs the image links so we can access them.
- `rasterio` reads raster imagery such as NAIP.
- `from_bounds` and `transform_bounds` help us clip the exact geographic area we want.


In [ ]:
from pystac_client import Client
import planetary_computer as pc
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

## Import the package for object detection

Now we add the machine learning package.

- `YOLO` from `ultralytics` loads a pretrained detection model and runs inference on images.
- in this notebook we use an OBB model, so detections come back in `result.obb` rather than `result.boxes`

See here for more information [https://github.com/ultralytics/ultralytics](https://github.com/ultralytics/ultralytics)


In [ ]:
from ultralytics import YOLO

## First set of helper functions: display an RGB image clearly

Satellite imagery often contains large data values. Before plotting, we usually stretch the image so it looks better on screen.

The first function below does a simple percentile stretch. The second function displays an image with a title.

In [ ]:
def stretch_rgb(rgb_bands):
    """Convert a 3-band image from bands-first to bands-last and stretch it for display."""
    rgb_hwc = np.moveaxis(rgb_bands, 0, -1).astype("float32")
    p2 = np.nanpercentile(rgb_hwc, 2)
    p98 = np.nanpercentile(rgb_hwc, 98)
    return np.clip((rgb_hwc - p2) / (p98 - p2 + 1e-6), 0, 1)


def show_image(img, title, figsize=(7, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()

## Second set of helper functions: search for a NAIP image

We now write a small function that searches the Planetary Computer for NAIP imagery.

The function does three things:
1. searches for NAIP scenes that intersect our bounding box
2. keeps the most recent date
3. chooses the scene with the greatest overlap with our area of interest


In [ ]:
def bbox_overlap_area(a, b):
    """Calculate the overlap area of two lon/lat bounding boxes."""
    minx = max(a[0], b[0])
    miny = max(a[1], b[1])
    maxx = min(a[2], b[2])
    maxy = min(a[3], b[3])
    if maxx <= minx or maxy <= miny:
        return 0.0
    return (maxx - minx) * (maxy - miny)


def get_latest_naip_item(bbox, datetime="2021-01-01/2024-12-31"):
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    items = list(
        catalog.search(
            collections=["naip"],
            bbox=bbox,
            datetime=datetime,
            limit=12,
            method="POST",
        ).items()
    )

    if not items:
        raise ValueError("No NAIP scenes found for the requested bbox and date range.")

    latest_date = max(item.datetime for item in items if item.datetime).date()
    same_day_items = [item for item in items if item.datetime and item.datetime.date() == latest_date]
    return max(same_day_items, key=lambda item: bbox_overlap_area(bbox, item.bbox))

## Third set of helper functions: clip the image to our study area

The NAIP image is stored in a projected coordinate system, but our bounding box is written in longitude and latitude.

So we have to:
1. convert the lon/lat bounding box into the raster's coordinate system
2. build a raster window from those bounds
3. read only the 3 RGB bands we need


In [ ]:
def read_naip_clip(item, bbox):
    href = item.assets["image"].href

    with rasterio.open(href) as src:
        minx, miny, maxx, maxy = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(minx, miny, maxx, maxy, src.transform)
        rgb = src.read([1, 2, 3], window=window)

    if rgb.shape[1] == 0 or rgb.shape[2] == 0:
        raise ValueError("The requested bbox produced an empty image. Try a different location.")

    return rgb

## Choose a place to analyze

We start with part of the Port of Baltimore because this scene returns many non-plane detections with the aerial OBB model, including ships and vehicles.

This gives us a better demonstration of the DOTA label set beyond the aircrafts we covered in the other notebook.


In [ ]:
site_name = "Port of Baltimore"
bbox = (-76.558, 39.258, -76.545, 39.263)


## Find the image and inspect its metadata

This cell searches the catalog and prints the date of the image we selected.

In [ ]:
item = get_latest_naip_item(bbox)
print("Selected site:", site_name)
print("Selected NAIP acquisition date:", item.datetime.date())
print(item.id)

## Read and display the clipped image

Now we read the image into memory, stretch it for display, and show it.

Notice the data shape before and after stretching:
- raw raster: `(bands, rows, cols)`
- display image: `(rows, cols, bands)`

In [ ]:
rgb = read_naip_clip(item, bbox)
rgb_display = stretch_rgb(rgb)

print("Raw raster shape:", rgb.shape)
print("Display image shape:", rgb_display.shape)
show_image(rgb_display, f"NAIP clip: {site_name}")

## Why do we tile the image?

Large aerial images are usually too large to process at full size with a detector.

We create smaller image tiles because:
- the model expects images near a standard size such as 640 by 640 pixels
- small targets like vehicles and aircraft are easier to detect if we do not shrink the whole scene too much
- tiling makes it easier to scale the workflow to larger study areas


## Fourth set of helper functions: create image tiles

The first small function calculates where each tile should begin. The second function actually cuts the image into tiles.

In [ ]:
def compute_positions(length, tile_size, overlap):
    step = max(tile_size - overlap, 1)
    if length <= tile_size:
        return [0]

    positions = list(range(0, length - tile_size + 1, step))
    if positions[-1] != length - tile_size:
        positions.append(length - tile_size)
    return positions


def make_tiles(image_hwc, tile_size=640, overlap=96):
    height, width, _ = image_hwc.shape
    y_positions = compute_positions(height, tile_size, overlap)
    x_positions = compute_positions(width, tile_size, overlap)

    tiles = []
    for y0 in y_positions:
        for x0 in x_positions:
            y1 = min(y0 + tile_size, height)
            x1 = min(x0 + tile_size, width)
            tile = image_hwc[y0:y1, x0:x1]
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1, "image": tile})
    return tiles

## Display a few example tiles

Before running the model, it is useful to inspect a few tiles and make sure they look reasonable.

In [ ]:
def plot_tiles(image_hwc, tiles, max_tiles=4):
    n = min(len(tiles), max_tiles)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, tile in zip(axes, tiles[:n]):
        ax.imshow(tile["image"])
        ax.set_title(f"x={tile['x0']} y={tile['y0']}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


tiles = make_tiles(rgb_display, tile_size=640, overlap=96)
print("Number of tiles:", len(tiles))
plot_tiles(rgb_display, tiles, max_tiles=4)

## Fifth set of helper functions: run YOLO OBB on each tile

This is the main detection function. It loops through the tiles, runs the pretrained OBB model, and stores the results in a pandas table.

A few details matter here:
- YOLO expects image-like values, so we convert each tile from `0-1` floats to `0-255` unsigned integers
- OBB models return detections in `result.obb`
- each detection gives us a class, a confidence score, an oriented box, and polygon corner coordinates
- we shift each polygon back into the full-image coordinate system using the tile offsets


In [ ]:
def run_yolo_obb_on_tiles(model, tiles, conf=0.2, imgsz=1024):
    rows = []

    for tile_id, tile in enumerate(tiles):
        tile_img = (tile["image"] * 255).clip(0, 255).astype("uint8")
        result = model.predict(tile_img, conf=conf, imgsz=imgsz, verbose=False)[0]
        obb = result.obb

        if obb is None or len(obb) == 0:
            continue

        xywhr = obb.xywhr.cpu().numpy()
        classes = obb.cls.cpu().numpy().astype(int)
        scores = obb.conf.cpu().numpy()
        polygons = obb.xyxyxyxy.cpu().numpy()

        for i in range(len(obb)):
            class_id = int(classes[i])
            score = float(scores[i])
            label = model.names[class_id]
            poly = polygons[i]
            shifted_poly = [[float(x + tile["x0"]), float(y + tile["y0"])] for x, y in poly]

            rows.append({
                "tile_id": tile_id,
                "label": label,
                "confidence": score,
                "cx": float(xywhr[i][0] + tile["x0"]),
                "cy": float(xywhr[i][1] + tile["y0"]),
                "width": float(xywhr[i][2]),
                "height": float(xywhr[i][3]),
                "angle_radians": float(xywhr[i][4]),
                "polygon": shifted_poly,
            })

    return pd.DataFrame(rows)


## Load the pretrained YOLO OBB model

We use `yolo26n-obb.pt`, which is a compact oriented bounding box model trained on DOTA aerial imagery classes.

If the weights are not already present, Ultralytics will download them automatically the first time you run this cell.


In [ ]:
model = YOLO("yolo26n-obb.pt")
print("Model task:", model.task)
print("Available classes:", model.names)


## Run detection and inspect the raw table

We now run the detector on all tiles. We focus first on non-plane labels that are common in port and industrial scenes.


In [ ]:
interesting_labels = {"ship", "harbor", "storage tank", "small vehicle", "large vehicle", "bridge"}

detections = run_yolo_obb_on_tiles(model, tiles, conf=0.2, imgsz=1024)

if detections.empty:
    print("No detections were returned. Try a different site, a larger tile size, or a lower confidence threshold such as 0.1.")
else:
    detections = detections[detections["label"].isin(interesting_labels)].copy()
    print("Relevant detections:", len(detections))
    display(detections.sort_values(["label", "confidence"], ascending=[True, False]).head(20))


## Plot the detections on the image

This final helper draws each oriented bounding box polygon on top of the original image.


In [ ]:
def plot_obb_detections(image_hwc, detections, keep_labels=None, figsize=(10, 10)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image_hwc)

    if keep_labels is not None:
        detections = detections[detections["label"].isin(keep_labels)].copy()

    colors = {
        "plane": "lime",
        "ship": "deepskyblue",
        "harbor": "orange",
        "bridge": "cyan",
        "large vehicle": "yellow",
        "small vehicle": "magenta",
        "storage tank": "white",
    }

    for _, row in detections.iterrows():
        color = colors.get(row["label"], "red")
        polygon = np.asarray(row["polygon"])

        patch = patches.Polygon(
            polygon,
            closed=True,
            linewidth=1.5,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(patch)

        x_text = float(np.min(polygon[:, 0]))
        y_text = float(np.min(polygon[:, 1]))
        ax.text(
            x_text,
            max(0, y_text - 4),
            f"{row['label']} {row['confidence']:.2f}",
            color="black",
            fontsize=8,
            bbox=dict(facecolor=color, alpha=0.85, pad=1),
        )

    ax.set_title("YOLO OBB detections")
    ax.axis("off")
    plt.show()


if not detections.empty:
    print(detections["label"].value_counts())
    plot_obb_detections(rgb_display, detections, keep_labels=interesting_labels, figsize=(12, 12))


## What should students notice?

Questions for discussion:

1. Which classes are detected most often in the scene?
2. Where do you see likely false positives?
3. Why might ships and vehicles be easier to detect here than buildings or roads?

Key point: aerial-imagery models can work well when the scene matches both the viewpoint and the label set used during training.


In [ ]:
# Example
# Step 1: Pick area of interest
site_name = "Houston Ship Channel"
bbox = (-95.092, 29.735, -95.050, 29.770)
interesting_labels = {"ship", "harbor", "storage tank"}

# Step 2: rerun the workflow
item = get_latest_naip_item(bbox)
rgb = read_naip_clip(item, bbox)
rgb_display = stretch_rgb(rgb)
tiles = make_tiles(rgb_display, tile_size=640, overlap=96)
detections = run_yolo_obb_on_tiles(model, tiles, conf=0.2, imgsz=1024)

if detections.empty:
    print("No detections returned for this site. Try another bbox or a lower confidence threshold.")
else:
    detections = detections[detections["label"].isin(interesting_labels)].copy()
    print("Site:", site_name)
    print("Image date:", item.datetime.date())
    print("Detections by class:")
    print(detections["label"].value_counts())
    show_image(rgb_display, f"New study area: {site_name}")
    plot_obb_detections(rgb_display, detections, keep_labels=interesting_labels, figsize=(12, 12))


## Guided exercise: apply the workflow to a new set of places of interest

Now students should reuse the code for new contexts using the aerial OBB model, but focus on scenes that match the DOTA class list.

Good targets include:
- ports and terminals
- harbors and marinas
- tank farms and refineries
- container yards
- large parking lots
- bridge crossings

Less suitable targets include:
- roads as continuous networks
- building footprints
- general urban land use

The task is to change the geographic bounding box to get images for a new area, and then rerun the workflow.

Write 200 words describing what the model detected in each scene you pick, what it missed, and why it may have performed that way.
